In [1]:
!pip install "transformers<5" torch sentencepiece
! pip install youtube-transcript-api
! pip install torch
!pip install faiss-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 12.8 MB/s eta 0:00:0000:01
   ━━━━━

In [2]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import faiss
qwen_model_name  = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer  = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model  = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [3]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')

    if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

urllink = input("Enter Url of the Video :\n")
if __name__ == "__main__":
    url = urllink

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id, languages=['en'])

    yttext = "\n".join(snippet.text for snippet in fetched)

print(yttext)

now stop me if you've heard this one
before but there are a lot of large
language models available today and they
have their own capabilities and
specialities what if I prefer to use one
llm to interpret some user queries in my
business application but a whole other
llm to author a response to those
queries well that scenario is exactly
what Lang chain caters to Lang chain is
an open-source orchestration framework
for the development of applications that
use large language models and it comes
in both Python and JavaScript libraries
it's it's essentially a generic
interface for nearly any llm so you have
a centralized development environment to
build your large language model
applications and then integrate them
with stuff like data sources and
software workflows now when it was
launched by Harrison Chase in October
2022 Lang chain enjoyed a meteoric rise
and by June of the following year it was
the single fastest growing open- source
project on GitHub and while the Lang
chain hype
trai

In [4]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
tokenizer = summarizer.tokenizer
model = summarizer.model 

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [5]:
tokenizer = summarizer.tokenizer
model = summarizer.model

inputs = tokenizer(
    yttext,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=1000,
    min_length=900,
    do_sample=False
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

Lang chain is an open-source orchestration framework for the development of applications that use large language models. Lang chain streamlines theprogramming of llm applications through something called abstractions. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. There's plenty of utility here so let's take a look at its components and see what Lang chain has to offer. It's essentially a genericinterface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows. There are also support for vectordatabases as well as traditional traditional datastructings by converting them into something called Vector embedings. It can also be used to split text up into small semantically meaningful chunks that can be combined into the form of text splitters which can 

In [6]:
answer = summarizer(summary, max_length=130, min_length=30, do_sample=False)
answer

[{'summary_text': "Lang chain is an open-source orchestration framework for the development of applications that use large language models. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. It's designed to provide a standard interface for all models so nearly any LM LM can be used in Lang chain."}]

In [7]:
print(answer[0]['summary_text'])

Lang chain is an open-source orchestration framework for the development of applications that use large language models. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. It's designed to provide a standard interface for all models so nearly any LM LM can be used in Lang chain.


In [ ]:
"""
def extract_text(text):
    full_text  = ""
    for i in text:
        full_text += extract_text() + "\n"
    return full_text
    """

'\ndef extract_text(text):\n    full_text  = ""\n    for i in text:\n        full_text += extract_text() + "\n"\n    return full_text\n    '

In [8]:
def chunks_text(yttext , chunk_size=500 , overlap=50):
    words = yttext.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [9]:
def clean_text(yttext):
    text = yttext.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [10]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [11]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [12]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [ ]:
""" GENERATE_TEXT_1
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = qwen_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    print(inputs["input_ids"].device)
    print(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=False,
        top_k=50,
        top_p=0.95,
    )
    return [qwen_tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
"""

In [13]:
def generate_text(prompt, max_new_tokens=300):
    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qwen_model.device)

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [14]:
text = clean_text(yttext)
chunks = chunks_text(yttext,chunk_size=120, overlap=30)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
Q = input("Enter your question :\n")
question = Q
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)
context = "\n\n".join(top_chunks)
for i, chunk in enumerate(top_chunks, 1):
    print(f"\n Answer {i} \n{chunk}")


 Answer 1 
and JavaScript libraries it's it's essentially a generic interface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows now when it was launched by Harrison Chase in October 2022 Lang chain enjoyed a meteoric rise and by June of the following year it was the single fastest growing open- source project on GitHub and while the Lang chain hype train has uh slightly cooled a little bit there's plenty of utility here so let's take a look at its components so what makes up Lang chain well Lang chain streamlines the programming of llm applications through something called abstractions

 Answer 2 
essentially Lang Chain's tools and apis simplify the process of building applications that make use of large language models if you have any questions please drop us a line below and if you want to see more videos like this in the future please like A

In [24]:
chunk = top_chunks[:3]
prompt = f"""
You are a question answering assistant.

Answer ONLY using the provided context and In short this is just the answer you need
If the answer is not in the context, reply exactly:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
answer = generate_text(prompt, max_new_tokens=300)
print(answer.strip())

Yes, Lang Chain is open source. It is mentioned that Lang Chain is "open-source" and "free to use". Additionally, it provides tools to monitor, evaluate, and debug applications, indicating its openness and accessibility. The context clearly states that Lang Chain is open source. Answer: Yes. I don't know.
